# 12. Convolution and vision blocks — faithful block anatomy

Only channel/image tensor sizes are reduced. Block order and stochastic-depth path are preserved.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")
print("device:", device)

## 1. Standard, grouped, depthwise-separable and dilated convolution

In [ ]:
x = torch.randn(1, 4, 16, 16, device=device)

standard_conv = nn.Conv2d(4, 8, 3, padding=1).to(device)
grouped_conv = nn.Conv2d(4, 8, 3, padding=1, groups=2).to(device)
depthwise_conv = nn.Conv2d(4, 4, 3, padding=1, groups=4).to(device)
pointwise_conv = nn.Conv2d(4, 8, 1).to(device)
dilated_conv = nn.Conv2d(4, 4, 3, padding=2, dilation=2).to(device)

print("standard:", standard_conv(x).shape)
print("grouped:", grouped_conv(x).shape)
print("depthwise-separable:", pointwise_conv(depthwise_conv(x)).shape)
print("dilated:", dilated_conv(x).shape)

## 2. Original post-activation ResNet bottleneck

The block keeps Conv-BN-ReLU / Conv-BN-ReLU / Conv-BN, projection shortcut when needed, residual addition, then final ReLU.

In [ ]:
class ResNetBottleneck(nn.Module):
    expansion = 4

    def __init__(
        self,
        input_channels=64,
        bottleneck_channels=16,
        stride=1,
    ):
        super().__init__()
        output_channels = bottleneck_channels * self.expansion

        self.conv1 = nn.Conv2d(
            input_channels,
            bottleneck_channels,
            1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(bottleneck_channels)

        self.conv2 = nn.Conv2d(
            bottleneck_channels,
            bottleneck_channels,
            3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(bottleneck_channels)

        self.conv3 = nn.Conv2d(
            bottleneck_channels,
            output_channels,
            1,
            bias=False,
        )
        self.bn3 = nn.BatchNorm2d(output_channels)

        if stride != 1 or input_channels != output_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    input_channels,
                    output_channels,
                    1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(output_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        hidden = F.relu(self.bn1(self.conv1(x)))
        hidden = F.relu(self.bn2(self.conv2(hidden)))
        hidden = self.bn3(self.conv3(hidden))
        return F.relu(hidden + identity)


resnet_block = ResNetBottleneck().to(device)
resnet_input = torch.randn(2, 64, 16, 16, device=device)
resnet_output = resnet_block(resnet_input)

assert resnet_block.expansion == 4
assert isinstance(resnet_block.bn3, nn.BatchNorm2d)
print("ResNet bottleneck:", resnet_output.shape)

## 3. ConvNeXt block including Layer Scale and Stochastic Depth

The previous notebook omitted stochastic depth. The original residual branch is 7×7 depthwise convolution → channel-last LayerNorm → 4× MLP → GELU → projection → layer scale → DropPath → residual add.

In [ ]:
class DropPath(nn.Module):
    def __init__(self, drop_probability=0.0):
        super().__init__()
        self.drop_probability = drop_probability

    def forward(self, x):
        if self.drop_probability == 0.0 or not self.training:
            return x

        keep_probability = 1.0 - self.drop_probability
        shape = (x.size(0),) + (1,) * (x.ndim - 1)
        random_tensor = keep_probability + torch.rand(
            shape,
            dtype=x.dtype,
            device=x.device,
        )
        binary_mask = random_tensor.floor()
        return x * binary_mask / keep_probability


class ConvNeXtBlock(nn.Module):
    def __init__(
        self,
        channels=32,
        layer_scale_init=1e-6,
        drop_path=0.1,
    ):
        super().__init__()

        self.depthwise = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )
        self.norm = nn.LayerNorm(channels, eps=1e-6)
        self.pointwise1 = nn.Linear(channels, 4 * channels)
        self.pointwise2 = nn.Linear(4 * channels, channels)
        self.gamma = nn.Parameter(
            layer_scale_init * torch.ones(channels)
        )
        self.drop_path = DropPath(drop_path)

    def forward(self, x):
        residual = x

        hidden = self.depthwise(x)
        hidden = hidden.permute(0, 2, 3, 1)
        hidden = self.norm(hidden)
        hidden = self.pointwise1(hidden)
        hidden = F.gelu(hidden)
        hidden = self.pointwise2(hidden)
        hidden = self.gamma * hidden
        hidden = hidden.permute(0, 3, 1, 2)
        hidden = self.drop_path(hidden)
        return residual + hidden


convnext_block = ConvNeXtBlock().to(device)
convnext_input = torch.randn(2, 32, 16, 16, device=device)
convnext_output = convnext_block(convnext_input)

assert convnext_block.depthwise.kernel_size == (7, 7)
assert convnext_block.depthwise.groups == 32
assert convnext_block.pointwise1.out_features == 4 * 32
assert isinstance(convnext_block.drop_path, DropPath)

print("ConvNeXt output:", convnext_output.shape)
print("DropPath probability:", convnext_block.drop_path.drop_probability)

## Structural checklist

ResNet and ConvNeXt now retain every demonstrated block component. No architectural branch is removed; only tensor widths and spatial sizes are small.